# Plan for Failures

Satellites and launches fail, and what failures cost depends on how the mission responds to them. This notebook simulates failures over each design's mission, and compares ways to restore failed satellites as decisions of each design.

It covers:

* a **failure model** of satellites and launches;
* **restoration policies**, as decisions of the design;
* choosing a design and its policy together;
* **attributing** the spread of the replicates to their uncertain parameters.

It builds on `PlanDeployment`: the study evaluates each history of failures by the phases in which its satellites operate.

## Model Failures

A `FailureModel` gives the life of a satellite from its launch, and the probability that a launch fails. The life here is a bathtub curve (`Bathtub`) of infant mortality, random failures, and wear-out (illustrative), and 5% of launches fail:

In [1]:
from tatc.tradespace import Bathtub, Exponential, FailureModel, Weibull

FAILURE_MODEL = FailureModel(
    life=Bathtub(infant=Weibull(shape=0.5, scale=100), random=Exponential(0.03), wear_out=Weibull(shape=4, scale=9)),
    launch=0.05,
)
print(f"{1 - FAILURE_MODEL.life.survival(5):.0%} of the satellites fail within 5 years")

37% of the satellites fail within 5 years


## Respond to Failures

A `RestorationPolicy` responds to failures. Here, five policies:

* **degrade**: fly the satellites that remain;
* **produce**: replace each failed satellite by producing a new one, after half a year of procurement, and launching it;
* **spare in orbit**: launch a spare with each plane, which takes a failed satellite's place within months;
* **spare on ground**: store a spare, which is launched at the next opportunity;
* **spares in orbit and on ground**: both.

A spare that is used is produced again. The policy is a decision of each design: `policy_space` declares whether to restore failed satellites, and the numbers of spares in each plane and on the ground (none or one of each). `space.extend` adds the policy to the design space, with the seed of the failures (`failure_seed`, drawn for each replicate).

`Failures` simulate 20 histories of the failures of each replicate, restored by the design's policy. The study evaluates each history by the phases in which its satellites operate, and gives the means over the histories and their quantiles: here, the share accurately sampled in the worst 10% of the histories (`accurate mission histories p10`), and the cost at 70% confidence (`cost histories p70`). The designs are the efficient designs by accuracy of `ExplainTradeSpace`, each with each policy.

In [2]:
from trade_space_example import ACCURACY, cost_space, cost_study, describe
from tatc.tradespace import (
    Failures,
    Integer,
    RestorationPolicy,
    Uncertain,
    aggregate,
    pareto_front,
    policy_space,
    summarize,
)

summary = summarize(cost_study.run(), ["satellites", "planes", "accurate"])
efficient = summary[pareto_front(summary, ACCURACY)]

POLICY = policy_space(orbit_spares=Integer(0, 1), ground_spares=Integer(0, 1), replenish=True)
failure_space = cost_space.extend(failure_seed=Uncertain.seed(), policy=POLICY)
FAILURES = Failures(
    FAILURE_MODEL,
    RestorationPolicy(procurement_years=0.5, replenish=True),  # the policy of each design starts from this one
    histories=20,
    seed="failure_seed",
    quantiles={"accurate mission": 0.1, "cost": 0.7},
)
failure_study = cost_study.replace(space=failure_space, failures=FAILURES)

# the efficient designs, each with each policy
with_policies, base_of = [], {}
for base in cost_space.designs_of(efficient):
    for policy in POLICY.enumerate():
        design = failure_space.design(base, policy=policy)
        with_policies.append(design)
        base_of[design.key] = base.key
trades = failure_study.run(with_policies)
print(f"{len(with_policies)} designs and policies x 5 replicates x 20 histories")

65 designs and policies x 5 replicates x 20 histories


The table gives four of the designs, each with each policy: two satellites in two planes, four in two, six in three, and nine in three. Its columns are the means over the replicates of the share accurately sampled over the mission (`accurate`, on average over the histories, and `accurate p10`, in their worst 10%) and of the cost at 70% confidence (`cost p70`). The first column is the share of the full constellation, without deployment or failures, from `ExplainTradeSpace`.

In [3]:
import pandas as pd


def describe_policy(row):
    """The name of a design's policy."""
    if not row["policy.restore"]:
        return "degrade"
    orbit, ground = row["policy.orbit_spares"] > 0, row["policy.ground_spares"] > 0
    return {
        (False, False): "produce",
        (True, False): "spare in orbit",
        (False, True): "spare on ground",
        (True, True): "spares in orbit and on ground",
    }[orbit, ground]


POLICIES = ["degrade", "produce", "spare in orbit", "spare on ground", "spares in orbit and on ground"]
# the means over the replicates of the mean share over the histories, the share in their worst 10%, and the cost at
# 70% confidence over the histories
ROBUST = {
    "accurate": "accurate mission",
    "accurate p10": "accurate mission histories p10",
    "cost p70": "cost histories p70",
}
traded = aggregate(trades, statistics=ROBUST).set_index("design")
traded["policy"] = pd.Categorical(traded.apply(describe_policy, axis=1), POLICIES)
traded["base"] = [base_of[key] for key in traded.index]
bases = efficient.set_index("design")
traded["description"] = bases.loc[traded.base].apply(describe, axis=1).to_numpy()
traded["full constellation"] = bases.loc[traded.base, "accurate"].to_numpy()

# four of the designs, with each policy: two satellites in two planes, four in two, six in three, and nine in three
shown = efficient.drop_duplicates(["satellites", "planes"]).set_index(["satellites", "planes"])
shown = shown.loc[[(2, 2), (4, 2), (6, 3), (9, 3)], "design"]
table = traded[traded.base.isin(shown)].sort_values(["satellites", "policy"])
table.round(1).set_index(["description", "full constellation", "policy"])[list(ROBUST)]

accurate  \
description                                        full constellation policy                                    
800 km, 60 deg FOR, Walker 45 deg: 2 x 1 (spaci... 34.2               degrade                            17.7   
                                                                      produce                            26.1   
                                                                      spare in orbit                     28.2   
                                                                      spare on ground                    28.9   
                                                                      spares in orbit and on ground      28.5   
800 km, 60 deg FOR, SSO: 2 at 15:00, 2 at 09:00    85.4               degrade                            50.7   
                                                                      produce                            68.5   
                                                                      spare in orbit                     68.8   
                                                                      spare on ground                    72.5   
                                                                      spares in orbit and on ground      69.3   
700 km, 60 deg FOR, Walker 75 deg: 3 x 2 (spaci... 96.4               degrade                            64.2   
                                                                      produce                            78.0   
                                                                      spare in orbit                     74.2   
                                                                      spare on ground                    79.6   
                                                                      spares in orbit and on ground      74.3   
800 km, 60 deg FOR, Walker 75 deg: 3 x 3 (spaci... 97.8               degrade                            79.3   
                                                                      produce                            82.8   
                                                                      spare in orbit                     76.8   
                                                                      spare on ground                    82.8   
                                                                      spares in orbit and on ground      76.9   

                                                                                                     accurate p10  \
description                                        full constellation policy                                        
800 km, 60 deg FOR, Walker 45 deg: 2 x 1 (spaci... 34.2               degrade                                 0.6   
                                                                      produce                                19.5   
                                                                      spare in orbit                         25.1   
                                                                      spare on ground                        25.6   
                                                                      spares in orbit and on ground          25.6   
800 km, 60 deg FOR, SSO: 2 at 15:00, 2 at 09:00    85.4               degrade                                10.8   
                                                                      produce                                56.2   
                                                                      spare in orbit                         61.3   
                                                                      spare on ground                        62.2   
                                                                      spares in orbit and on ground          63.1   
700 km, 60 deg FOR, Walker 75 deg: 3 x 2 (spaci... 96.4               degrade                                38.9   
                                                                      produce                

The shares over the mission fall short of those of the full constellation under every policy, by the phases of deployment and the failures of launches as well as of satellites:

* **Degrading** is cheapest, but loses the most, above all in the worst histories: two satellites in two planes keep 18% on average, and 1% in their worst 10% of histories, as both can fail. Nine satellites keep the most of what they would sample (79% of 98%), as a failure leaves eight.
* **Producing replacements** restores most of what degrading loses, at 17 to 34% more cost.
* **A spare on the ground** restores as well or better, for about 16 million dollars more than producing replacements, and pays off in the worst histories of the small designs: two satellites in two planes keep 26% in their worst histories with it, against 20%.
* **A spare in orbit can hurt.** Each plane waits for its spare to be produced, and a spare in a plane of three satellites takes a second launch, which delays the deployment. Six satellites in three planes then sample less on average than by producing replacements (74% against 78%), and nine satellites less than by degrading (77% against 79%), at a higher cost. Spares both in orbit and on the ground cost the most, and sample about as much as a spare in orbit alone.

## Choose a Design and its Policy

The front of cost at 70% confidence against the share accurately sampled in the worst 10% of the histories chooses designs and policies together:

In [4]:
chosen = traded[pareto_front(traded, {"cost p70": "min", "accurate p10": "max"})]
chosen.sort_values("cost p70")[["description", "policy", "accurate", "accurate p10", "cost p70"]].round(1)

,description,policy,accurate,accurate p10,cost p70
design,,,,,
2a310c00151b,"800 km, 60 deg FOR, SSO: 1 at 12:00",degrade,3.2,0.4,110.2
bc2076a0d3d8,"800 km, 60 deg FOR, SSO: 2 at 06:00",degrade,7.0,2.1,128.9
bb6574900ac3,"800 km, 60 deg FOR, SSO: 1 at 12:00",produce,3.9,3.5,130.6
638697713902,"800 km, 60 deg FOR, Walker 75 deg: 1 x 3 (spac...",degrade,10.9,4.6,146.2
43f50c8bc38a,"800 km, 60 deg FOR, SSO: 2 at 06:00",produce,9.2,7.3,153.4
ad99c42082ab,"800 km, 60 deg FOR, Walker 45 deg: 2 x 1 (spac...",produce,26.1,19.5,159.4
cc2e14e787ff,"800 km, 60 deg FOR, Walker 45 deg: 2 x 1 (spac...",spare on ground,28.9,25.6,175.2
eeb7c70a720b,"800 km, 60 deg FOR, SSO: 2 at 15:00, 1 at 09:00",produce,47.3,38.5,198.1
3a902b976460,"700 km, 60 deg FOR, Walker 75 deg: 3 x 2 (spac...",degrade,64.2,38.9,209.5


Along the front, degrading is the cheapest policy of a design, and restoring buys robustness. Two satellites in two Walker planes keep 20% of the points accurately sampled in their worst histories if they produce replacements (159 million dollars), and 26% with a spare on the ground (175 million). Six satellites in three Walker planes keep 39%, 70%, and 72% if they degrade, produce replacements, or keep a spare on the ground, for 210, 279, and 294 million. Nine satellites in three planes degrade gracefully (69% in their worst histories, for 254 million), and producing replacements raises them to 79%, for 340 million. A spare on the ground is efficient for designs of two to six satellites, but a spare in orbit is not efficient for any design. The policy that a design deserves depends on the design, which is why it is a decision of the design.

## Find What Spreads the Results

The replicates of a design differ by its uncertain parameters: the sample of points (`points_seed`), the histories of failures (`failure_seed`), and the growth of the cost of development (`cost_growth`). Which of them spreads an objective says what would sharpen it. The sample of points and the histories are noise of the estimates, which more points or more histories reduce; the growth of costs is a risk of the mission.

`attribute_variance` evaluates each design again with one parameter drawn as in other replicates and the others held. The **total share** of a parameter is the share of the variance between replicates that it causes, alone or with others (near 0 if it does not matter), with its standard error (`se`). Here, for the designs and policies of the front, from 20 replicates, 20 more, and 20 with each parameter drawn as in the second 20:

In [5]:
import time

from tatc.tradespace import attribute_variance

began = time.perf_counter()
attribution = attribute_variance(
    failure_study, list(ROBUST.values()), failure_space.designs_of(trades, chosen.index), replicates=20
)
print(f"{len(attribution.results)} evaluations in {time.perf_counter() - began:.0f} s")
attribution.summary().mul(100).round(1)

1700 evaluations in 0 s


,accurate mission,accurate mission histories p10,cost histories p70,accurate mission se,accurate mission histories p10 se,cost histories p70 se
parameter,,,,,,
points_seed,92.4,30.4,0.0,6.4,3.3,0.0
cost_growth,0.0,0.0,81.8,0.0,0.0,19.0
failure_seed,3.6,67.0,35.8,0.7,6.5,7.1


The sample of points spreads the mean shares accurately sampled (92% of their variance between replicates), and the failures hardly do (4%), as each replicate averages 20 histories. The share in the worst 10% of the histories is spread mostly by the failures (67%, and 30% by the points), as a quantile of 20 histories is a noisy estimate. Both are noise of the estimates, with different remedies: more points would sharpen the means, and more histories the worst 10%, which is cheap.

The cost at 70% confidence is spread by the growth of the cost of development (82%), a risk of the mission, and by the failures (36%), which cost replacements and spares. These shares are the least certain, with standard errors of 19 and 7 points. The total shares of each objective sum to about 100% within their errors, so the parameters hardly interact.

The order of designs on the front depends on their differences, in which the parameters that act alike on two designs cancel. `differences` attributes the variance of the differences between pairs of designs from the same evaluations: here, between the designs and policies adjacent along the front, by cost:

In [6]:
ordered = chosen.sort_values("cost p70").index
attribution.differences(list(zip(ordered, ordered[1:]))).summary().mul(100).round(1)

,accurate mission,accurate mission histories p10,cost histories p70,accurate mission se,accurate mission histories p10 se,cost histories p70 se
parameter,,,,,,
points_seed,92.1,30.4,0.0,6.4,4.5,0.0
cost_growth,0.0,0.0,0.0,0.0,0.0,0.0
failure_seed,2.9,73.0,105.8,0.5,6.5,4.7


The growth of the cost of development cancels in every difference. All of these designs fly the same imager, whose development grows alike for each, so the growth is a risk of each mission but not a reason to prefer one design to another; the failures alone spread the differences of their costs. The shares differ by what spreads them: the sample of points for the means (92%), and mostly the histories for the worst 10% (73%). More points would settle the order of neighbors by their means, and more histories their order by the worst 10% and by cost.

## Summary

* A `FailureModel` draws the failures of satellites and launches, and a `RestorationPolicy` restores them (by producing new satellites, or by spares in orbit or on the ground) or not. `Failures` simulate histories of each replicate's mission, for both the cost and the capability of each policy.
* The policy is a decision of each design (`policy_space`), which the front chooses with the design.
* `attribute_variance` finds the uncertain parameters that spread the replicates of each design, which says what would sharpen each objective, and `Attribution.differences` those that spread the differences between designs.

This is the last notebook of the tutorial.

## Notes

* Simulating failures is cheap, but evaluating each phase's satellites is not, and most phases repeat the same sets of satellites. The histories of a replicate are therefore evaluated together, with a cache that evaluates each set of satellites once (`evaluate_missions`, `PhaseCache`).
* The histories are stratified: each satellite's life and each launch is drawn from its own stratum of its distribution, which estimates means several times more precisely than independent histories (`Deployment.simulate_many`).
* Without restoration (degrading), satellites fail independently, so the mean share over the failures is computed exactly, from the probability that each set of satellites operates (`expected_missions`); its worst 10% and its cost still come from the histories.
* A design life does not retire a satellite, and a replacement whose launch fails is not flown again (see `DeploymentSchedule`).